# Sprint Block M — Pakistan local track

Adds a Pakistan-specific evaluation and analysis on top of the frozen global pipeline
(Blocks H + I). **Nothing here touches `results_h/` or the 106-sentence base-paper comparison.**

| Step | What | Runtime |
|---|---|---|
| **M1** | Bbox-filter the existing 1M corpus → how much Pakistan data do we already have? | CPU |
| M2 | New collection (Reddit / Wikipedia / OSM) — done by a separate collection agent | — |
| M3 | Annotate + merge the new PK data (reuse `PlaceInterpreter.pipe()` + I1 `weak_label()`) | GPU if >2k |
| M4 | Ingest the hand-annotated PK gold evaluation set | CPU |
| M5 | Region-specific experiments: transfer / local-only / merged | CPU |

Design rule: one `annotated_v2.parquet` with a `region` column, sliceable global vs pk.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, glob
ROOT = '/content/drive/MyDrive/geoproject'
os.environ['HF_HOME'] = f'{ROOT}/models_cache/hf'

# Block M writes to results_m/. Reads annotated_v1.parquet + eval/ + results_i/.
os.makedirs(f'{ROOT}/results_m', exist_ok=True)

print('ROOT =', ROOT)
print('geoproject/ contains:', sorted(os.listdir(ROOT)))


## M1 — How much Pakistan data is already in the corpus?

The corpus is 99.8% `source='wikivoyage'`, but Wikivoyage has Karachi / Lahore / Islamabad /
Peshawar / Rawalpindi articles — those sentences are already collected, cleaned and annotated,
just not labelled as Pakistani. M1 finds them by **coordinate**, not by `source`.

Steps:
1. Check lat/lon coverage in `annotated_v1.parquet`. If < 90% have coords, a bbox filter would
   silently miss rows → fall back to matching `place_name` against a Pakistani city list.
2. Filter to the Pakistan bounding box (lat 23.5–37.1, lon 60.9–77.8).
3. Trim border noise (the bbox clips into Afghanistan / India / Iran) with a coarse polygon test.
4. Report the count, the source and anchor-quality breakdown, and the weak-tag distribution.
5. Write `results_m/pk_from_corpus.parquet` with `region='pk'`.

**Decision gate:** if this beats the 1,200 rows that `source in (osm_pk, commons_pk)` gives, the
PK *training* side is already viable and M2 only needs to build the evaluation set.


In [ ]:
import pandas as pd, numpy as np

annot = pd.read_parquet(f'{ROOT}/annotated/annotated_v1.parquet')
print(f'annotated_v1.parquet: {len(annot):,} rows')

has_coord = annot.lat.notna() & annot.lon.notna()
print(f'rows with lat AND lon : {has_coord.sum():,}  ({has_coord.mean():.1%})')

# --- real Pakistan boundary (Natural Earth admin-0, via a public mirror) -----
import urllib.request, json as _json
try:
    import geopandas as gpd
except ImportError:
    os.system('pip -q install geopandas shapely')
    import geopandas as gpd
from shapely.geometry import Point
from shapely.prepared import prep

_gj = f'{ROOT}/results_m/_pakistan.geojson'
if not os.path.exists(_gj):
    url = ('https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/'
           'geojson/ne_10m_admin_0_countries.geojson')
    urllib.request.urlretrieve(url, f'{ROOT}/results_m/_world.geojson')
    w = gpd.read_file(f'{ROOT}/results_m/_world.geojson')
    pk_geom = w[w.NAME.str.contains('Pakistan', case=False, na=False)].geometry.iloc[0]
    gpd.GeoDataFrame(geometry=[pk_geom], crs='EPSG:4326').to_file(_gj, driver='GeoJSON')
pk_shape = prep(gpd.read_file(_gj).geometry.iloc[0].buffer(0.03))   # ~3 km tolerance

cand = annot[has_coord].copy()
in_pk = np.fromiter((pk_shape.contains(Point(x, y))
                     for x, y in zip(cand.lon.values, cand.lat.values)),
                    bool, len(cand))
cand = cand[in_pk]
print(f'inside Pakistan boundary : {len(cand):,}')

# --- belt-and-braces: drop rows whose Wikivoyage place_name is clearly Indian
INDIA = ('(india)', 'himachal', 'jammu', 'kashmir valley', 'rajasthan', 'chandigarh',
         'ladakh', 'punjab (india)', 'haryana', 'delhi', 'udaipur', 'shimla', 'manali',
         'dharamsala', 'srinagar', 'jodhpur', 'jaisalmer', 'chittorgarh', 'amritsar')
_pn = cand.place_name.fillna('').str.lower()
drop_in = _pn.apply(lambda s: any(k in s for k in INDIA))
pk = cand[~drop_in].copy()
pk['region'] = 'pk'
print(f'after Indian-place-name filter: {len(pk):,}  (dropped {drop_in.sum():,})')

print(f'\n{"="*60}\nPK-from-corpus: {len(pk):,} sentences\n{"="*60}')
print('\nby source:');        print(pk.source.value_counts().to_string())
print('\nby anchor_quality:'); print(pk.anchor_quality.value_counts().to_string())
print('\ntop place_name:');    print(pk.place_name.value_counts().head(20).to_string())
print(f'\nvs source-tag PK (osm_pk+commons_pk): {annot.source.isin(["osm_pk","commons_pk"]).sum():,}')

# --- weak tags (exact I1 rule) --------------------------------------------
import sys; sys.path.insert(0, ROOT)
import adjective_extractor as ax
LEXICON = set(json.load(open(f'{ROOT}/eval/evalset_106.json'))['lexicon'])
def _n2t(noun, osm_key):
    if osm_key and '=' in osm_key:
        v = osm_key.split('=')[-1].strip().lower().replace('_', '-')
        if v in LEXICON: return v
    n = (noun or '').strip().lower().replace('_', '-')
    return n if n in LEXICON else None
NOUN2TAG = {n: _n2t(n, s[0]) for n, s in ax.OSM_NOUN_LEXICON.items()}
EXT = {k: v for k, v in {**{k: v for k, v in NOUN2TAG.items() if v},
       'bar':'bar','pub':'pub','brewery':'brewery','gallery':'gallery','bookshop':'bookshop',
       'mall':'mall','zoo':'zoo','theatre':'theatre','cinema':'cinema','library':'library',
       'pharmacy':'pharmacy','bank':'bank','supermarket':'supermarket','cathedral':'cathedral',
       'chapel':'chapel','monument':'monument','tomb':'tomb','tower':'tower','palace':'palace',
       'ruins':'ruins','fountain':'fountain','monastery':'monastery'}.items() if v in LEXICON}

def weak_label(row):
    ents = json.loads(row['entities']) if isinstance(row['entities'], str) else row['entities']
    hints = json.loads(row['tag_hints']) if isinstance(row['tag_hints'], str) else row['tag_hints']
    for e in ents:
        if e.get('category') == 'unnamed':
            t = _n2t(e.get('head_lemma') or e.get('lemma', ''), e.get('osm_key'))
            if t: return t
    for h in hints:
        v = str(h.get('value', '')).lower().replace('_', '-')
        if v in LEXICON: return v
    for e in ents:
        lem = (e.get('head_lemma') or e.get('lemma') or '').lower()
        if lem in EXT: return EXT[lem]
    return None

pk['weak_tag'] = pk.apply(weak_label, axis=1)
print(f'\nweak-labelled: {pk.weak_tag.notna().sum():,} / {len(pk):,} '
      f'({pk.weak_tag.notna().mean():.0%})')
print('\ntop weak tags:'); print(pk.weak_tag.value_counts().head(20).to_string())

pk.to_parquet(f'{ROOT}/results_m/pk_from_corpus.parquet', index=False)
print(f'\nwrote results_m/pk_from_corpus.parquet')


In [ ]:
# sanity check: are these actually Pakistani places?
pk = pd.read_parquet(f'{ROOT}/results_m/pk_from_corpus.parquet')
print('top place_name values (should be PK cities/regions):')
print(pk.place_name.value_counts().head(25).to_string())
print('\nsample sentences:')
for _, r in pk[pk.weak_tag.notna()].sample(8, random_state=1).iterrows():
    print(f"  [{r.weak_tag}] ({r.lat:.2f},{r.lon:.2f}) {r.text[:130]}")
# how many distinct places / cities
print(f'\ndistinct place_name: {pk.place_name.nunique():,}')


## M3a — Add the `region` column to the full corpus (merge, no re-annotation)

The 4,928 PK sentences are *already in* `annotated_v1.parquet` — they were annotated in Block D.
So the "merge" is just stamping `region='pk'` on those `sent_id`s and `region='global'` on the
rest, producing `annotated_v2.parquet`. Same for the weak-labelled training set. When M2's new
data arrives it gets annotated (M3b) and concatenated onto v2.


In [ ]:
pk_ids = set(pd.read_parquet(f'{ROOT}/results_m/pk_from_corpus.parquet', columns=['sent_id']).sent_id)

# annotated_v2 = v1 + region column
av2 = annot.copy()
av2['region'] = np.where(av2.sent_id.isin(pk_ids), 'pk', 'global')
av2.to_parquet(f'{ROOT}/annotated/annotated_v2.parquet', index=False)
print(f'annotated_v2.parquet: {len(av2):,} rows  '
      f'({(av2.region=="pk").sum():,} pk / {(av2.region=="global").sum():,} global)')

# weak-labelled training set + region (reuse I1 output)
wl = pd.read_parquet(f'{ROOT}/results_i/weak_labelled.parquet')
wl['region'] = np.where(wl.sent_id.isin(pk_ids), 'pk', 'global')
wl.to_parquet(f'{ROOT}/results_m/weak_labelled_v2.parquet', index=False)
print(f'weak_labelled_v2.parquet: {len(wl):,} rows  '
      f'({(wl.region=="pk").sum():,} pk / {(wl.region=="global").sum():,} global)')
print(f'  pk rows by split: {wl[wl.region=="pk"].split.value_counts().to_dict()}')

json.dump({'pk_from_corpus': int(len(pk_ids)),
           'pk_weak_labelled': int((wl.region=="pk").sum()),
           'pk_weak_train': int(((wl.region=="pk") & (wl.split=="train")).sum()),
           'source_tag_pk_baseline': int(annot.source.isin(["osm_pk","commons_pk"]).sum()),
           'bbox': 'Natural Earth PK boundary + Indian-place denylist',
           'decision': 'PK training data viable from existing corpus; M2 only needs the eval set'},
          open(f'{ROOT}/results_m/m1_summary.json', 'w'), indent=2)
print('\nwrote results_m/m1_summary.json')
